## Student Starter Pack

This notebook is a starting template for class projects using **ArtBench-10**.

It covers:

1. Loading ArtBench-10 from the local folder `artbench_generative_suite/ArtBench-10`
2. Exploring dataset shape and class distribution
3. Building PyTorch dataloaders
4. Visualizing samples in a grid
5. Exporting samples to image files (one image per file)
6. Loading subset definitions from `training.csv` generated by `generate_training_csv.py`


## Dataset quick notes

- **Domain**: paintings / artistic styles
- **Classes**: 10 styles
- **Image size**: 32x32 RGB
- **Splits**: train and test

In this project setup, dataset files are expected in:

- `ArtBench-10/artbench-10-python/artbench-10-batches-py/`
- `ArtBench-10/ArtBench-10.csv`

If you do not have it on the folder, download from kaggle directly:

https://www.kaggle.com/datasets/alexanderliao/artbench10


### Setup e Configurações
Definição dos caminhos absolutos, hiperparâmetros globais (como Sementes Aleatórias para garantir reprodutibilidade) e configurações específicas da arquitetura VAE.

In [ ]:
# Import libraries
from __future__ import annotations
import json
from collections import Counter

from torch.utils.data import DataLoader

import random
import numpy as np
import torch
from torch_fidelity import calculate_metrics
import matplotlib.pyplot as plt
import importlib

# Import / reload files
import config as cfg
importlib.reload(cfg)

import models.Variational_Autoenconders
importlib.reload(models.Variational_Autoenconders)

import models.Gans
importlib.reload(models.Gans)

import models.PixelUNet
importlib.reload(models.PixelUNet)

import utils.io
importlib.reload(utils.io)

import utils.visualize
importlib.reload(utils.visualize)

import utils.metrics
importlib.reload(utils.metrics)

import data.artbench_local_dataset
importlib.reload(data.artbench_local_dataset)

import data.dataloader
importlib.reload(data.dataloader)

# Import functions
from data.artbench_local_dataset import load_kaggle_artbench10_splits
from data.dataloader import HFDatasetTorch, get_transforms_vae, get_transforms_gan

from utils.io import safe_num_workers, make_subset_indices, load_ids_from_training_csv, export_split_to_folder, save_experiment_results
from utils.visualize import show_batch_grid, plot_gan_losses, show_image_grid, plot_curve
from utils.metrics import evaluate_vae, interpolacao_latente_vae, gerar_amostras_fid_vae, gerar_amostras_fid_gan, interpolacao_latente_gan, gerar_amostras_fid_diffusion, interpolacao_latente_diffusion ,run_inference, mostrar_reconstrucoes, gerar_grelha_amostras, extrair_amostras_reais_fid

from models.Variational_Autoenconders import VAE, train_vae
from models.Gans import init_dcgan_weights, DCDiscriminator, DCGenerator, train_gan, save_checkpoint
from models.PixelUNet import train_diffusion, GaussianDiffusion, PixelUNet


In [ ]:
# Define seed
random.seed(cfg.SEED)
np.random.seed(cfg.SEED)
torch.manual_seed(cfg.SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.SEED)

# Define device
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Device: {device}")

# Define Number os workers
EFFECTIVE_NUM_WORKERS = safe_num_workers(cfg.NUM_WORKERS)

# Create base folders
cfg.KAGGLE_ROOT.mkdir(parents=True, exist_ok=True)
cfg.EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
cfg.VAE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
cfg.GAN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
cfg.DIFFUSION_RESULTS_DIR.mkdir(parents=True, exist_ok=True)


### Preparação do Dataset (ArtBench-10)
Carregamento do *subset* de 20% do dataset de treino. As imagens são redimensionadas para 32x32 e normalizadas para o intervalo [0, 1] para alimentar o modelo.

In [ ]:
hf_ds = load_kaggle_artbench10_splits(cfg.KAGGLE_ROOT)
train_hf = hf_ds["train"]

print("Train size:", len(train_hf))
print("Columns   :", train_hf.column_names)

label_feature = train_hf.features["label"]
class_names = list(label_feature.names)
num_classes = len(class_names)

print("Num classes:", num_classes)
print("Class names:", class_names)

### Check dataset distribution

In [ ]:
# Class distribution summary
train_counts = Counter(train_hf["label"])

print("\nTrain class distribution:")
for cid, name in enumerate(class_names):
    print(f"  {cid:2d} | {name:>15s} | {train_counts.get(cid, 0):6d}")

### Build PyTorch datasets and dataloaders

In [ ]:
transform = get_transforms_vae(cfg.IMAGE_SIZE)

train_indices = make_subset_indices(len(train_hf), cfg.TRAIN_FRACTION, seed=cfg.SEED)

train_ds = HFDatasetTorch(train_hf, transform=transform, indices=train_indices)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=EFFECTIVE_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Train dataset length (after fraction):", len(train_ds))
print("Train batches                        :", len(train_loader))

### Load subset of 20% samples `training_20_percent.csv` 

you can reproduce the same subset in this notebook by loading IDs from that CSV.

Use `train_id_original` for indexing this notebook's full train split.


In [ ]:
train_ids_from_csv = load_ids_from_training_csv(cfg.CSV_PATH_20, index_column=cfg.INDEX_COLUMN)

print('Loaded ids:', len(train_ids_from_csv))
print('First 10 ids:', train_ids_from_csv[:10])

# Build a train dataset/loader using exactly those IDs
train_ds_from_csv = HFDatasetTorch(train_hf, transform=transform, indices=train_ids_from_csv)
train_loader_from_csv = DataLoader(
    train_ds_from_csv,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=EFFECTIVE_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print('Subset train dataset length:', len(train_ds_from_csv))
print('Subset train batches      :', len(train_loader_from_csv))


### Visualize a sample grid

In [ ]:
show_batch_grid(train_loader, class_names, n_images=36, nrow=6, title='ArtBench-10 Train Samples')

### Export samples to image files

This helper saves one PNG per sample and writes a CSV with metadata.

In [60]:
export_split_to_folder(train_loader, class_names, cfg.EXPORT_ROOT / 'train_subset', max_images=500)

Exported 500 images to: c:\Users\Asus\Desktop\Git\master-projects\gen-ai-project\src\..\data\train_subset\images
Metadata CSV: c:\Users\Asus\Desktop\Git\master-projects\gen-ai-project\src\..\data\train_subset\metadata.csv


### Arquitetura do Variational Autoencoder (VAE)
Construção do modelo base com:
- **Encoder:** Redução espacial simétrica (32 -> 16 -> 8 -> 4).
- **Reparameterization Trick:** Mapeamento para o espaço latente ($\mu$ e $\sigma$).
- **Decoder:** Reconstrução visual da imagem.
- **Função de Loss:** Combinação de Reconstrução (BCE) e Regularização Latente (KL Divergence).

In [ ]:
# Recriar o test_loader para garantir que ele entrega (imagem, label, id)
test_dataset = HFDatasetTorch(hf_ds["test"], transform=transform)
test_loader = DataLoader(test_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False)

# Inicializar o Modelo e o Otimizador
model_vae = VAE(latent_dim=cfg.VAE_LATENT_DIM).to(device)

if cfg.VAE_OPTIM == "ADAM":
    optimizer = torch.optim.Adam(model_vae.parameters(), lr=cfg.VAE_LR)
else:
    optimizer = torch.optim.AdamW(model_vae.parameters(), lr=cfg.VAE_LR, weight_decay=1e-4)

# Executar o Treino
print("\nA iniciar o treino do VAE no dataset...")
hist_vae = train_vae(
    model=model_vae, 
    loader=train_loader_from_csv if cfg.ONLY_20 else train_loader, 
    optimizer=optimizer, 
    device=device,       
    epochs=cfg.N_EPOCHS, 
    beta=cfg.VAE_BETA
)

print("Treino concluído!")

### Avaliação, Resultados Visuais e Métricas (FID/KID)
- Extração das métricas finais de teste
- Geração da **Interpolação no Espaço Latente** (Avaliação qualitativa).
- Geração de 5000 imagens aleatórias para cálculo estatístico.
- Cálculo matemático da distância das distribuições (**FID e KID**) utilizando a biblioteca `torch-fidelity`.
- Exportação automática de todos os pesos, configs, métricas e gráficos para a pasta respetiva da *run*.


In [ ]:
# AVALIAÇÃO DO MODELO 

print("\nA avaliar o modelo nos dados de teste...")
metrics_vae = evaluate_vae(
    model=model_vae, 
    loader=test_loader, 
    device=device, 
    beta=cfg.VAE_BETA
)

print('\nResultados Finais do VAE')
print(f"Loss Total: {metrics_vae['loss']:.4f}")
print(f"Reconstrução (BCE): {metrics_vae['recon_bce']:.4f}")
print(f"KL Divergence: {metrics_vae['kl']:.4f}")

configs_usadas = {
    "model": "VAE",
    "latent_dim": cfg.VAE_LATENT_DIM,
    "beta": cfg.VAE_BETA,
    "epochs": cfg.N_EPOCHS,
    "batch_size": cfg.BATCH_SIZE,
    "seed": cfg.SEED
}

save_experiment_results(
    run_dir=cfg.VAE_RESULT_PATH,
    model=model_vae,
    history=hist_vae,
    config_dict=configs_usadas,
    test_metrics=metrics_vae
)

# GERAÇÕES VISUAIS E FID
print("\n🎨 A gerar interpolação visual...")
batch = next(iter(test_loader))
imagens_reais = batch[0]
img_A = imagens_reais[0] 
img_B = imagens_reais[1] 

# Guarda o gráfico de interpolação!
interpolacao_latente_vae(model_vae, img_A, img_B, device, steps=8, save_path=cfg.VAE_RESULT_PATH / "interpolacao.png")

### Visualização das Reconstruções (VAE)
Geração de uma grelha comparativa entre as imagens reais do conjunto de teste (em cima) e as respetivas reconstruções feitas pelo VAE (em baixo), permitindo avaliar visualmente a qualidade do modelo.

In [ ]:
print("\nA visualizar reconstruções...")
mostrar_reconstrucoes(
    modelo=model_vae, 
    test_loader=test_loader, 
    device=device,
    num_imagens=8,
    save_path=cfg.VAE_RESULT_PATH / "reconstrucoes.png"
)

### Geração de Amostras Aleatórias (VAE)
Amostragem de ruído no espaço latente para gerar 16 obras de arte completamente novas a partir do zero, demonstrando a capacidade generativa e criativa do modelo.

In [ ]:
print("\nA gerar artes completamente novas...")
gerar_grelha_amostras(
    modelo=model_vae, 
    device=device,
    num_samples=16,
    save_path=cfg.VAE_RESULT_PATH / "amostras_aleatorias.png"
)

### Extração de Amostras Reais (FID)
Extração de 5.000 imagens reais para servirem como base de referência (Ground Truth) no cálculo matemático das métricas de qualidade (FID e KID) das imagens geradas.

In [ ]:
# Extrair 5000 imagens reais do train_loader
extrair_amostras_reais_fid(
    loader=train_loader_from_csv if cfg.ONLY_20 else train_loader, 
    num_samples=cfg.EVALUATE_SAMPLE_N, 
    output_dir=cfg.PASTA_REAIS_FID
)

### Avaliação Estatística (Protocolo de 10 Runs)
Execução de um protocolo de avaliação rigoroso que gera e avalia 5.000 imagens ao longo de 10 iterações independentes (com *seeds* diferentes). Este processo calcula a média e o desvio padrão das métricas FID e KID, garantindo a robustez e a validade estatística dos resultados finais do VAE.

In [ ]:
# Configurações da Avaliação Estatística
resultados_finais = []

print(f"A iniciar Protocolo de Avaliação para VAE...")

for i in range(cfg.EVALUATE_N_RUNS):
    # 1. Definir a Seed única para esta run
    current_seed = cfg.SEED + i 
    torch.manual_seed(current_seed)
    np.random.seed(current_seed)
    
    # 2. Criar pasta específica para esta run
    run_path = cfg.VAE_PASTA_FID / f"run_{i+1:02d}"
    run_path.mkdir(parents=True, exist_ok=True)
    
    print(f"\n--- [RUN {i+1:02d}/{cfg.EVALUATE_N_RUNS}] Seed: {current_seed} ---")
    print(f"-> Pasta: {run_path}")

    # 3. Gerar as 5000 amostras para esta pasta específica
    gerar_amostras_fid_vae(model_vae, cfg.EVALUATE_SAMPLE_N, cfg.BATCH_SIZE, device, run_path)

    # 4. Calcular Métricas comparando com as Reais
    metricas = calculate_metrics(
        input1=str(cfg.PASTA_REAIS_FID),  
        input2=str(run_path),        
        cuda=False,
        isc=False,                    
        fid=True,                     
        kid=True,
        kid_subset_size=100,
        kid_subsets=50, 
        verbose=False 
    )

    # 5. Guardar JSON individual desta run dentro da sua pasta
    with open(run_path / "metrics.json", "w") as f:
        json.dump(metricas, f, indent=4)

    # 6. Acumular valores para estatística final
    resultados_finais.append({
        'run': i + 1,
        'seed': current_seed,
        'fid': metricas['frechet_inception_distance'],
        'kid': metricas['kernel_inception_distance_mean']
    })
    
    print(f"FID: {metricas['frechet_inception_distance']:.4f} | KID: {metricas['kernel_inception_distance_mean']:.5f}")

# CÁLCULO DAS ESTATÍSTICAS FINAIS

fids = [r['fid'] for r in resultados_finais]
kids = [r['kid'] for r in resultados_finais]

estatisticas = {
    "fid_mean": float(np.mean(fids)),
    "fid_std": float(np.std(fids)),
    "kid_mean": float(np.mean(kids)),
    "kid_std": float(np.std(kids))
}

print(f"FID Final: {estatisticas['fid_mean']:.4f} ± {estatisticas['fid_std']:.4f}")
print(f"KID Final: {estatisticas['kid_mean']:.5f} ± {estatisticas['kid_std']:.4f}")

# Guardar o relatório consolidado final
with open(cfg.VAE_PASTA_FID / "final_statistical_report.json", "w") as f:
    json.dump({"runs": resultados_finais, "summary": estatisticas}, f, indent=4)

print(f"\nResultados guardados em: {cfg.VAE_PASTA_FID / 'final_statistical_report.json'}")

### Preparação dos Dados (GAN & Difusão)
Configuração dos DataLoaders com a normalização matemática exigida por estas arquiteturas (píxeis no intervalo `[-1, 1]`). Instancia dois cenários de treino: o conjunto de dados completo e o subconjunto restrito de 20% para a análise de eficiência de dados.

In [ ]:
# Transformações exclusivas da GAN [-1, 1]
transform_gan = get_transforms_gan(cfg.IMAGE_SIZE)

# 1. DataLoader do Dataset Completo
train_ds_gan = HFDatasetTorch(train_hf, transform=transform_gan, indices=train_indices)
train_loader_gan_dm = DataLoader(
    train_ds_gan, batch_size=cfg.BATCH_SIZE, shuffle=True, 
    num_workers=EFFECTIVE_NUM_WORKERS, pin_memory=torch.cuda.is_available()
)

# 2. DataLoader do Subset de 20% do CSV
train_ds_gan_from_csv = HFDatasetTorch(train_hf, transform=transform_gan, indices=train_ids_from_csv)
train_loader_gan_dm_from_csv = DataLoader(
    train_ds_gan_from_csv, batch_size=cfg.BATCH_SIZE, shuffle=True, 
    num_workers=EFFECTIVE_NUM_WORKERS, pin_memory=torch.cuda.is_available()
)

### Inicialização e Treino da DCGAN
Instanciação das redes do Gerador e Discriminador, seguida da inicialização de pesos customizada (um passo crucial para garantir a estabilidade matemática inicial). Por fim, executa o *loop* de treino adversarial e regista o histórico de perdas (*losses*).

In [ ]:
# Inicializar o Modelo
netG = DCGenerator(latent_dim=cfg.GAN_LATENT_DIM, image_channels=3).to(device)
netD = DCDiscriminator(image_channels=3).to(device)

# Inicializar pesos (Obrigatório para estabilidade)
netG.apply(init_dcgan_weights)
netD.apply(init_dcgan_weights)

# Executar o Treino
print(f"\nA iniciar o treino da DCGAN no dataset ArtBench por {cfg.N_EPOCHS} épocas...")
hist_gan = train_gan(
    generator=netG,
    discriminator=netD,
    loader=train_loader_gan_dm_from_csv if cfg.ONLY_20 else train_loader_gan_dm, 
    latent_dim=cfg.GAN_LATENT_DIM,
    epochs=cfg.N_EPOCHS,
    lr=cfg.GAN_LR,
    device=device
)
print("Treino concluído!")

### Exportação de Resultados e Curvas de Treino (GAN)
Guarda os pesos finais do Gerador e do Discriminador, exporta os hiperparâmetros de configuração para formato JSON e gera o gráfico das *Losses* para visualização da estabilidade e equilíbrio adversarial.

In [ ]:
print("\nA guardar resultados da experiência GAN...")

# Guardar o modelo
save_checkpoint(
    generator=netG,
    discriminator=netD,
    history=hist_gan,
    checkpoint_path=cfg.GAN_RESULT_PATH / "gan_model.pt",
    latent_dim=cfg.GAN_LATENT_DIM,
    channels=3,
    image_size=cfg.IMAGE_SIZE
)

# Guardar configurações
configs_gan = {
    "model": "DCGAN",
    "latent_dim": cfg.GAN_LATENT_DIM,
    "epochs": cfg.N_EPOCHS,
    "batch_size": cfg.BATCH_SIZE,
    "learning_rate": cfg.GAN_LR,
    "seed": cfg.SEED
}
with open(cfg.GAN_RESULT_PATH / "config_gan.json", "w") as f:
    json.dump(configs_gan, f, indent=4)

# Gerar e guardar gráfico
plot_gan_losses(hist_gan, title='DCGAN Losses - ArtBench', save_path=cfg.GAN_RESULT_PATH / "loss_curve.png")

### Geração de Novas Amostras e Interpolação Latente (GAN)
Amostragem de ruído aleatório para gerar 16 obras de arte completamente novas a partir do zero. De seguida, executa uma interpolação no espaço latente (*Latent Walk*) para visualizar a transição suave entre duas imagens distintas, provando que a GAN aprendeu um espaço de características contínuo e não apenas memorizou o dataset original.

In [ ]:
print("\nA gerar amostras novas (Random Latent Sampling)...")
run_inference(netG, cfg.GAN_LATENT_DIM, num_samples=16, device=device, save_path=cfg.GAN_RESULT_PATH / "amostras_aleatorias.png")

print("\nA gerar interpolação visual...")
interpolacao_latente_gan(netG, cfg.GAN_LATENT_DIM, steps=8, device=device, save_path=cfg.GAN_RESULT_PATH / "interpolacao.png")

### Avaliação Estatística (Protocolo de 10 Runs - DCGAN)
Execução de um protocolo de avaliação. O modelo gera e avalia 5.000 imagens ao longo de 10 iterações independentes (variando as *seeds*). O cálculo da média e do desvio padrão das métricas FID e KID garante a robustez estatística e comprova a verdadeira performance do modelo adversarial.

In [ ]:
# Configurações do Protocolo de Avaliação
resultados_gan = []

print(f"A iniciar Protocolo de Avaliação para DCGAN...")

for i in range(cfg.EVALUATE_N_RUNS):
    # 1. Definir a Seed única para esta run
    current_seed = cfg.SEED + i 
    torch.manual_seed(current_seed)
    np.random.seed(current_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(current_seed)
    
    # 2. Criar subpasta para esta run
    run_path = cfg.GAN_PASTA_FID / f"run_{i+1:02d}"
    run_path.mkdir(parents=True, exist_ok=True)
    
    print(f"\n--- [RUN {i+1:02d}/{cfg.EVALUATE_N_RUNS}] Seed: {current_seed} ---")
    print(f"-> Gerando imagens em: {run_path}")

    # 3. Gerar as 5000 amostras usando o Gerador da GAN
    gerar_amostras_fid_gan(
        netG, 
        cfg.EVALUATE_SAMPLE_N, 
        cfg.BATCH_SIZE, 
        cfg.GAN_LATENT_DIM, 
        device, 
        run_path
    )

    # 4. Calcular Métricas comparando com as Imagens Reais
    metricas = calculate_metrics(
        input1=str(cfg.PASTA_REAIS_FID),  
        input2=str(run_path),        
        cuda=False,
        isc=False,                    
        fid=True,                     
        kid=True,
        kid_subset_size=100,
        kid_subsets=50, 
        verbose=False 
    )

    # 5. Guardar JSON individual desta
    with open(run_path / "metrics_gan.json", "w") as f:
        json.dump(metricas, f, indent=4)

    # 6. Acumular valores para o relatório final
    resultados_gan.append({
        'run': i + 1,
        'seed': current_seed,
        'fid': metricas['frechet_inception_distance'],
        'kid': metricas['kernel_inception_distance_mean']
    })
    
    print(f"FID: {metricas['frechet_inception_distance']:.4f} | KID: {metricas['kernel_inception_distance_mean']:.5f}")

# CÁLCULO DAS ESTATÍSTICAS

fids = [r['fid'] for r in resultados_gan]
kids = [r['kid'] for r in resultados_gan]

resumo_estatistico = {
    "fid_mean": float(np.mean(fids)),
    "fid_std": float(np.std(fids)),
    "kid_mean": float(np.mean(kids)),
    "kid_std": float(np.std(kids))
}

print(f"FID Final: {resumo_estatistico['fid_mean']:.4f} ± {resumo_estatistico['fid_std']:.4f}")
print(f"KID Final: {resumo_estatistico['kid_mean']:.5f} ± {resumo_estatistico['kid_std']:.4f}")

# Guardar o relatório final
with open(cfg.CAMINHO_GAN_FID_JSON, "w") as f:
    json.dump({"runs": resultados_gan, "summary": resumo_estatistico}, f, indent=4)

print(f"\nRelatório estatístico da GAN guardado com sucesso!")

### Inicialização e Treino do Modelo de Difusão (DDPM)
Configuração do processo matemático de difusão (escalonamento de ruído) e instanciação da arquitetura `PixelUNet`. De seguida, executa o *loop* de treino onde o modelo aprende iterativamente a remover o ruído gaussiano para reconstruir as obras de arte do conjunto de dados.

In [ ]:
# 1) Instanciar o processo de difusão
pixel_diffusion = GaussianDiffusion(num_timesteps=cfg.DIFF_TIMESTEPS, device=device)

# 2) Instanciar a UNet (Cuidado: in_channels=3 para RGB)
pixel_model = PixelUNet(in_channels=3, model_channels=cfg.DIFF_CHANNELS).to(device)
optimizer_diff = torch.optim.Adam(pixel_model.parameters(), lr=cfg.DIFF_LR)

# 3) Executar o Treino
print(f"\n🚀 A iniciar o treino da Difusão (PixelUNet) no dataset ArtBench")

hist_diff = train_diffusion(
    model=pixel_model, 
    loader=train_loader_gan_dm_from_csv if cfg.ONLY_20 else train_loader_gan_dm, 
    schedule=pixel_diffusion, 
    epochs=cfg.N_EPOCHS, 
    lr=cfg.DIFF_LR,
    device=device
)

print("Treino concluído!")

### Geração de Novas Amostras e Interpolação (Modelo de Difusão)
Amostragem completa do processo de difusão reverso: partindo de puro ruído gaussiano estatístico, o modelo gera uma grelha de 16 obras de arte inteiramente novas. Em seguida, é efetuada uma interpolação esférica (Slerp) no espaço de ruído inicial para visualizar a transição orgânica e gradual entre dois conceitos artísticos gerados pelo modelo.

In [ ]:
print("\nA gerar artes completamente novas (Random Noise Sampling)...")

# Gerar 16 imagens aleatórias
shape_random = (16, 3, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE) 
sampled_random = pixel_diffusion.p_sample_loop(pixel_model, shape_random)

show_image_grid(
    images=sampled_random, 
    channels=3, 
    title='Artes Geradas do Zero (Difusão)', 
    n_show=16,
    save_path=cfg.DIFFUSION_RESULT_PATH / "amostras_aleatorias.png"
)

print("\nA gerar interpolação visual (Slerp)...")
interpolacao_latente_diffusion(
    diffusion_schedule=pixel_diffusion, 
    model=pixel_model, 
    device=device, 
    steps=8, 
    save_path=cfg.DIFFUSION_RESULT_PATH / "interpolacao.png"
)

### Exportação de Resultados e Curvas de Treino (Modelo de Difusão)
Preservação integral da experiência: guarda os pesos finais da rede `PixelUNet` num *checkpoint* (incluindo metadados), exporta os hiperparâmetros para formato JSON e gera o gráfico do Erro Quadrático Médio (*MSE Loss*) para analisar a convergência do treino.

In [ ]:
print("\nA guardar resultados da experiência de Difusão...")

# Guardar o modelo
torch.save({
    'epoch': cfg.N_EPOCHS,
    'model_state_dict': pixel_model.state_dict(),
    'channels': 3,
    'image_size': cfg.IMAGE_SIZE
}, cfg.DIFFUSION_RESULT_PATH / "diffusion_model.pt")

# Guardar configurações
configs_diff = {
    "model": "PixelUNet_DDPM",
    "timesteps": cfg.DIFF_TIMESTEPS,
    "epochs": cfg.N_EPOCHS,
    "batch_size": cfg.BATCH_SIZE,
    "learning_rate": 2e-4,
    "seed": cfg.SEED
}
with open(cfg.DIFFUSION_RESULT_PATH / "config_diffusion.json", "w") as f:
    json.dump(configs_diff, f, indent=4)

# Gerar e guardar gráfico
plot_curve(hist_diff, title='Pixel-space Diffusion Loss - ArtBench', ylabel='MSE Loss', save_path=cfg.DIFFUSION_RESULT_PATH / "loss_curve.png")

### Avaliação Estatística (Protocolo de 10 Runs - Difusão)
Execução do protocolo de avaliação estatística final para o Modelo de Difusão. Ao longo de 10 iterações com *seeds* independentes, o modelo gera 5.000 imagens através do seu processo iterativo de remoção de ruído (*timesteps*). O cálculo da média e do desvio padrão das métricas FID e KID consolida a performance definitiva desta arquitetura.

In [ ]:
# Configurações do Protocolo de Avaliação
resultados_diff = []

print(f"A iniciar Protocolo de Avaliação para Difusão...")

for i in range(cfg.EVALUATE_N_RUNS):
    # 1. Definir a Seed única para esta run
    current_seed = cfg.SEED + i 
    torch.manual_seed(current_seed)
    np.random.seed(current_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(current_seed)
    
    # 2. Criar subpasta para esta run
    run_path = cfg.DIFFUSION_PASTA_FID / f"run_{i+1:02d}"
    run_path.mkdir(parents=True, exist_ok=True)
    
    print(f"\n--- [RUN {i+1:02d}/{cfg.EVALUATE_N_RUNS}] Seed: {current_seed} ---")
    print(f"-> Gerando imagens em: {run_path}")

    # 3. Gerar as 5000 amostras
    gerar_amostras_fid_diffusion(
        diffusion_schedule=pixel_diffusion, 
        model=pixel_model, 
        num_samples=cfg.EVALUATE_SAMPLE_N, 
        batch_size=cfg.BATCH_SIZE,
        device=device, 
        output_dir=run_path
    )

    # 4. Calcular Métricas comparando com as Reais
    metricas = calculate_metrics(
        input1=str(cfg.PASTA_REAIS_FID),  
        input2=str(run_path),        
        cuda=False,
        isc=False,                    
        fid=True,                     
        kid=True,
        kid_subset_size=100,
        kid_subsets=50, 
        verbose=False 
    )

    # 5. Guardar JSON individual desta run
    with open(run_path / "metrics_diff.json", "w") as f:
        json.dump(metricas, f, indent=4)

    # 6. Acumular valores para o cálculo de média/desvio padrão
    resultados_diff.append({
        'run': i + 1,
        'seed': current_seed,
        'fid': metricas['frechet_inception_distance'],
        'kid': metricas['kernel_inception_distance_mean']
    })
    
    print(f"Run {i+1} Concluída -> FID: {metricas['frechet_inception_distance']:.4f} | KID: {metricas['kernel_inception_distance_mean']:.5f}")

# CÁLCULO DAS ESTATÍSTICAS

fids = [r['fid'] for r in resultados_diff]
kids = [r['kid'] for r in resultados_diff]

resumo_estatistico = {
    "fid_mean": float(np.mean(fids)),
    "fid_std": float(np.std(fids)),
    "kid_mean": float(np.mean(kids)),
    "kid_std": float(np.std(kids))
}

print(f"FID Final: {resumo_estatistico['fid_mean']:.4f} ± {resumo_estatistico['fid_std']:.4f}")
print(f"KID Final: {resumo_estatistico['kid_mean']:.5f} ± {resumo_estatistico['kid_std']:.4f}")

# Guardar o relatório consolidado
with open(cfg.DIFFUSION_RESULT_PATH / "final_statistical_report.json", "w") as f:
    json.dump({"runs": resultados_diff, "summary": resumo_estatistico}, f, indent=4)

print(f"\nRelatório estatístico da Difusão concluído com sucesso!")